# 04 — Verify LiteRT/TFLite model trên local Windows (v2)

Bản này sửa lỗi **fixed batch = 1** của các model LiteRT đã export.

Các `.tflite` hiện có input dạng:
- 320 model: `[1, 3, 320, 320]`
- 640 model: `[1, 3, 640, 640]`

Vì batch dimension bị cố định là `1`, không được đưa một list 8 ảnh vào `model.predict()` cùng lúc và không được validate với `batch=16`.

Notebook này:
1. kiểm tra model/file;
2. inspect tensor;
3. smoke test từng ảnh một;
4. so `.pt` với `.tflite` cho YOLO11n-640;
5. full validation với `batch=1`;
6. so metric LiteRT với metric `.pt`.


## 1. Kiểm tra môi trường

In [2]:
import sys, platform
print("Python:", sys.version)
print("OS:", platform.platform())

import ultralytics
print("Ultralytics:", ultralytics.__version__)

try:
    import ai_edge_litert
    print("ai-edge-litert: OK")
except ImportError:
    print("ai-edge-litert: CHƯA CÀI")


Python: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
OS: Windows-10-10.0.19045-SP0
Ultralytics: 8.4.142
ai-edge-litert: OK


### Nếu thiếu `ai-edge-litert`

Bỏ comment dòng dưới, chạy một lần rồi restart kernel.


In [3]:
# %pip install -U ai-edge-litert

## 2. Xác định project và model

In [4]:
from pathlib import Path

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path(r"D:\Project\TrafficSignAI"),
]

PROJECT_ROOT = next(
    (p for p in candidate_roots if (p / "VR-TSD-2").exists() and (p / "models").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy project TrafficSignAI.")

TRAINED_DIR = PROJECT_ROOT / "models" / "trained"
DEPLOY_DIR = PROJECT_ROOT / "models" / "deploy"
TEST_IMAGES_DIR = PROJECT_ROOT / "VR-TSD-2" / "test" / "images"
DATA_YAML = PROJECT_ROOT / "VR-TSD-2" / "data_local.yaml"

MODELS = {
    "yolo11n_320": {
        "pt": TRAINED_DIR / "yolo11n_320_best.pt",
        "tflite": DEPLOY_DIR / "yolo11n_320.tflite",
        "imgsz": 320,
    },
    "yolo11n_640": {
        "pt": TRAINED_DIR / "yolo11n_640_best.pt",
        "tflite": DEPLOY_DIR / "yolo11n_640.tflite",
        "imgsz": 640,
    },
    "yolo11s_320": {
        "pt": TRAINED_DIR / "yolo11s_320_best.pt",
        "tflite": DEPLOY_DIR / "yolo11s_320.tflite",
        "imgsz": 320,
    },
    "yolo11s_640": {
        "pt": TRAINED_DIR / "yolo11s_640_best.pt",
        "tflite": DEPLOY_DIR / "yolo11s_640.tflite",
        "imgsz": 640,
    },
}

print("Project root:", PROJECT_ROOT)
for name, spec in MODELS.items():
    print(
        name,
        "| PT:", spec["pt"].exists(),
        "| TFLite:", spec["tflite"].exists(),
        "| imgsz:", spec["imgsz"],
    )


Project root: D:\Project\TrafficSignAI
yolo11n_320 | PT: True | TFLite: True | imgsz: 320
yolo11n_640 | PT: True | TFLite: True | imgsz: 640
yolo11s_320 | PT: True | TFLite: True | imgsz: 320
yolo11s_640 | PT: True | TFLite: True | imgsz: 640


## 3. Kiểm tra labels.txt

In [5]:
LABELS_PATH = DEPLOY_DIR / "labels.txt"
labels = [x.strip() for x in LABELS_PATH.read_text(encoding="utf-8").splitlines() if x.strip()]

print("Số class:", len(labels))
assert len(labels) == 58, f"Expected 58 classes, got {len(labels)}"


Số class: 58


## 4. Inspect tensor contract

In [6]:
from ai_edge_litert.interpreter import Interpreter

for name, spec in MODELS.items():
    print("\n" + "=" * 70)
    print(name)

    interpreter = Interpreter(model_path=str(spec["tflite"]))
    interpreter.allocate_tensors()

    for x in interpreter.get_input_details():
        print("INPUT :", x["shape"], x["dtype"])

    for x in interpreter.get_output_details():
        print("OUTPUT:", x["shape"], x["dtype"])



yolo11n_320
INPUT : [  1   3 320 320] <class 'numpy.float32'>
OUTPUT: [   1   62 2100] <class 'numpy.float32'>

yolo11n_640
INPUT : [  1   3 640 640] <class 'numpy.float32'>
OUTPUT: [   1   62 8400] <class 'numpy.float32'>

yolo11s_320
INPUT : [  1   3 320 320] <class 'numpy.float32'>
OUTPUT: [   1   62 2100] <class 'numpy.float32'>

yolo11s_640
INPUT : [  1   3 640 640] <class 'numpy.float32'>
OUTPUT: [   1   62 8400] <class 'numpy.float32'>


## 5. Chọn 8 ảnh test cố định

Các ảnh này sẽ được xử lý **từng ảnh một** vì model LiteRT có batch cố định bằng 1.


In [7]:
import random

image_files = sorted([
    p for p in TEST_IMAGES_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
])

random.seed(42)
sample_images = random.sample(image_files, min(8, len(image_files)))

print("Số ảnh test tổng:", len(image_files))
for p in sample_images:
    print("-", p.name)


Số ảnh test tổng: 1151
- ch0_20250430105000_20250430105300_f00288_jpg.rf.ff911426361b58657908d0bc8641a681.jpg
- 4_mp4-0020_jpg.rf.d83658bfc8daacc6130291fd9389449e.jpg
- Eddy2612-online-video-cutter_com-_mp4-0049_jpg.rf.4e4cdb7b8067b123efdef52d0ea245e1.jpg
- ch0_20250517120708_20250517121208_f03978_jpg.rf.a7db81db2c1cf46e5e6b87bba22c8372.jpg
- ch0_20250504163332_20250504163632_f04020_jpg.rf.5fea7fffc3a908d01593f97b003a8404.jpg
- ch0_20250430105600_20250430105900_f00324_jpg.rf.7770b43363a67dd5e5c03f0954cf687f.jpg
- ch0_20250430104659_20250430104959_f02394_jpg.rf.b066fb535c3445bf2158d28431a01b03.jpg
- NO20250710-172749-000012F_f03055_jpg.rf.f2a3a7c462bb3ff5e0896afd3f26c397.jpg


## 6. Smoke inference cả 4 `.tflite`

**Quan trọng:** không truyền list 8 ảnh vào một lần. Mỗi lần predict chỉ có `1` ảnh để khớp input `[1, 3, H, W]`.


In [8]:
from ultralytics import YOLO
import pandas as pd

SMOKE_DIR = PROJECT_ROOT / "runs" / "litert_smoke"
smoke_summary = []

for name, spec in MODELS.items():
    print("\n" + "=" * 70)
    print("SMOKE:", name)

    model = YOLO(str(spec["tflite"]))
    total_detections = 0
    processed = 0

    for image_path in sample_images:
        results = model.predict(
            source=str(image_path),
            imgsz=spec["imgsz"],
            conf=0.25,
            device="cpu",
            save=True,
            verbose=False,
            project=str(SMOKE_DIR),
            name=name,
            exist_ok=True,
        )

        processed += len(results)
        total_detections += sum(len(r.boxes) for r in results)

    smoke_summary.append({
        "model": name,
        "imgsz": spec["imgsz"],
        "images": processed,
        "detections": total_detections,
    })

    print("Images:", processed)
    print("Total detections:", total_detections)
    print("Saved to:", SMOKE_DIR / name)

smoke_df = pd.DataFrame(smoke_summary)
display(smoke_df)



SMOKE: yolo11n_320
Loading D:\Project\TrafficSignAI\models\deploy\yolo11n_320.tflite for LiteRT inference...
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320
Images: 8
Total detections: 8
Saved to: D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_320

SMOKE: yolo11n_640
Loading D:\Project\TrafficSignAI\models\deploy\yolo11n_640.tflite for LiteRT inference...
Results saved to D:\Project\TrafficSignAI\runs\litert_smoke\yolo11n_640
Results saved to D:\Project\TrafficSi

,model,imgsz,images,detections
0,yolo11n_320,320,8,8
1,yolo11n_640,640,8,11
2,yolo11s_320,320,8,9
3,yolo11s_640,640,8,11


## 7. So trực quan `.pt` và `.tflite` của YOLO11n-640

Cả hai cũng được chạy từng ảnh một để phép so đơn giản và đồng nhất.


In [9]:
MAIN_NAME = "yolo11n_640"
spec = MODELS[MAIN_NAME]

pt_model = YOLO(str(spec["pt"]))
tflite_model = YOLO(str(spec["tflite"]))

compare_dir = PROJECT_ROOT / "runs" / "pt_vs_litert_smoke"

for image_path in sample_images:
    _ = pt_model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        device=0,
        save=True,
        verbose=False,
        project=str(compare_dir),
        name="pt",
        exist_ok=True,
    )

    _ = tflite_model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        device="cpu",
        save=True,
        verbose=False,
        project=str(compare_dir),
        name="tflite",
        exist_ok=True,
    )

print("PT predictions     :", compare_dir / "pt")
print("TFLite predictions :", compare_dir / "tflite")


Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\pt
Loading D:\Project\TrafficSignAI\models\deploy\yolo11n_640.tflite for LiteRT inference...
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\tflite
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\pt
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\tflite
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\pt
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\tflite
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\pt
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\tflite
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\pt
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\tflite
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\pt
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_litert_smoke\tflite
Results saved to D:\Project\TrafficSignAI\runs\pt_vs_liter

## 8. Full validation LiteRT

Do model export với `batch=1`, validation cũng phải dùng `batch=1`.

Mặc định chỉ validate candidate chính `YOLO11n-640`.


In [13]:
MODELS_TO_VALIDATE = [
    "yolo11n_640",
    "yolo11n_320",
    "yolo11s_320",
    "yolo11s_640",
]

validation_rows = []

for name in MODELS_TO_VALIDATE:
    spec = MODELS[name]

    print("\n" + "=" * 70)
    print("FULL VALIDATION:", name)

    model = YOLO(str(spec["tflite"]))

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=spec["imgsz"],
        batch=1,
        device="cpu",
        workers=4,
        plots=True,
        project=str(PROJECT_ROOT / "runs" / "litert_validation"),
        name=name,
        exist_ok=True,
    )

    validation_rows.append({
        "experiment": name,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)



FULL VALIDATION: yolo11n_640
Ultralytics 8.4.142  Python-3.13.2 torch-2.14.0+cu126 CPU (13th Gen Intel Core i5-13400F)
Loading D:\Project\TrafficSignAI\models\deploy\yolo11n_640.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.00.0 ms, read: 1077.6279.8 MB/s, size: 141.1 KB)
val: Scanning D:\Project\TrafficSignAI\VR-TSD-2\test\labels.cache... 1151 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1151/1151 402.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1151/1151 17.2it/s 1:070.1ss
                   all       1151       1938      0.913      0.832       0.92       0.75
102-cam-di-nguoc-chieu        213        225       0.97      0.708      0.895      0.708
          103a-cam-oto         31         31      0.913      0.871      0.966      0.725
  103b-cam-oto-re-phai          1          1       0.68          1      0.995      0.895
  103c-cam-oto-

,experiment,precision,recall,mAP50,mAP50_95
0,yolo11n_640,0.913202,0.832230,0.920290,0.749624
1,yolo11n_320,0.780523,0.660411,0.729657,0.577848
2,yolo11s_320,0.914394,0.846487,0.897511,0.708333
3,yolo11s_640,0.935172,0.942058,0.970428,0.806390


## 9. So metric `.tflite` với `.pt`

In [14]:
# Tìm overnight_summary.csv trong toàn bộ project
summary_candidates = list(PROJECT_ROOT.rglob("overnight_summary.csv"))

print("Các file tìm thấy:")
for p in summary_candidates:
    print(p)

if not summary_candidates:
    print("Không tìm thấy overnight_summary.csv.")
else:
    summary_path = summary_candidates[0]
    pt_summary = pd.read_csv(summary_path)

    wanted = [
        "experiment",
        "test_precision",
        "test_recall",
        "test_mAP50",
        "test_mAP50_95",
    ]

    if all(col in pt_summary.columns for col in wanted):
        comparison = validation_df.merge(
            pt_summary[wanted],
            on="experiment",
            how="left",
        )

        comparison["delta_precision"] = comparison["precision"] - comparison["test_precision"]
        comparison["delta_recall"] = comparison["recall"] - comparison["test_recall"]
        comparison["delta_mAP50"] = comparison["mAP50"] - comparison["test_mAP50"]
        comparison["delta_mAP50_95"] = comparison["mAP50_95"] - comparison["test_mAP50_95"]

        display(comparison)
    else:
        print("overnight_summary.csv không có đúng bộ cột mong đợi.")


Các file tìm thấy:
D:\Project\TrafficSignAI\overnight_summary.csv


,experiment,precision,recall,mAP50,mAP50_95,test_precision,test_recall,test_mAP50,test_mAP50_95,delta_precision,delta_recall,delta_mAP50,delta_mAP50_95
0,yolo11n_640,0.913202,0.832230,0.920290,0.749624,0.922132,0.843105,0.929526,0.756673,-0.008930,-0.010875,-0.009236,-0.007049
1,yolo11n_320,0.780523,0.660411,0.729657,0.577848,0.766237,0.709237,0.741643,0.577623,0.014286,-0.048826,-0.011987,0.000224
2,yolo11s_320,0.914394,0.846487,0.897511,0.708333,0.917825,0.851263,0.901483,0.709645,-0.003431,-0.004775,-0.003972,-0.001312
3,yolo11s_640,0.935172,0.942058,0.970428,0.806390,0.950797,0.924849,0.969630,0.804516,-0.015625,0.017209,0.000798,0.001874


## Sau khi notebook này chạy ổn

Nếu `yolo11n_640.tflite` load được, predict được và metric gần `.pt`, bước tiếp theo là đưa model sang Android Studio:

`CameraX → preprocess → LiteRT → decode [1,62,8400] → NMS → bounding box thật`
